## Getting Started with PyleoTUPS: Automated Paleoclimate Data Extraction

**PyleoTUPS** is a Python package designed to streamline paleoclimate data workflows. It automates the extraction and processing of datasets from major repositories like NOAA NCEI.

Key features include:
* **Automated Extraction:** Parses complex text files and templates automatically.
* **Metadata Preservation:** Keeps study-level metadata (authors, location) linked to the data tables.
* **FAIR Compliance:** Supports standards like LiPD and NOAA PaST vocabularies.

In this tutorial, we will cover:
1.  **Searching** for studies using the NOAA API.
2.  **Browsing** search results (Metadata, Sites, Publications).
3.  **Extracting** data into pandas DataFrames.

In [1]:
import pyleotups as pt

# Initialize the Dataset manager
# This object will act as your session, storing search results and processing data.
dataset = pt.Dataset()

### Step 1: Searching for Studies

The primary entry point for finding data is the `search_studies()` method. This connects directly to the NOAA NCEI Paleo Search API.

```Note
Given parameter's are based on the endpoints by NCEI-NOAA and it's description can be found at https://www.ncei.noaa.gov/access/paleo-search/api
```

### 1.1 Search by Identifier
If you know the specific ID of a study, you can fetch it directly. 
*Note: When an ID is provided, all other search parameters (like location or author) are ignored.*

In [2]:
# Fetch a specific study by its NOAA ID
# This fills the 'dataset' object with the result.
dataset.search_studies(noaa_id=18316)

# View a summary of the result
dataset.get_summary()

[2025-12-11 10:28:33,466][INFO] - search_studies: Using identifier-only fetch (xml_id/NOAAStudyId). Other parameters will be ignored.


Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?NOAAStudyId=18316&dataPublisher=NOAA


Parsing NOAA studies: 100%|██████████| 1/1 [00:00<00:00, 1582.76it/s]


,StudyID,XMLID,StudyName,DataType,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding
0,18316,16019,"WAIS Divide Core 1,200 Year Atmospheric CO2 an...",ICE CORES,1217,11,733,1939,CO2 concentration and Stable Isotopic Composit...,[carbon cycle],"Thomas Bauska, Fortunat Joos, Alan Mix, Raphae...","[{'Author': 'Ahn, J., E. J. Brook, L. Mitchell...","[[{'DataTableID': '28694', 'DataTableName': 'W...",[{'fundingAgency': 'US National Science Founda...


### Search by Metadata

For broader discovery, you can search using a combination of parameters. `PyleoTUPS` supports lists for fields like investigators, keywords, and locations.

**Key Parameters:**
* **Geography:** `min_lat`, `max_lat`, `min_lon`, `max_lon`.
* **Investigators:** Provide a list of names. 
    * *Convention:* NOAA expects `"LastName, Initials"` (e.g., `"Wahl, E.R."`). Searching by just `"LastName"` often works but is less precise.
    * *Logic:* By default, multiple items are treated as **OR** (match *any*). Use `investigators_and_or="AND"` to match *all*.
* **Species:** Requires strict **4-letter codes** (e.g., `"PIPO"`. Find Species list [here](https://www.ncei.noaa.gov/access/paleo-search/study/params.json) ) .
* **Time:** `earliest_year`, `latest_year`.
* **Keywords:** Hierarchies are supported (e.g., `"earth science>paleoclimate"`).

##### Efficient Use of `investigator` as a search parameter

In [ ]:
# Clear previous results
dataset = pt.Dataset()

# Example 1: "OR" Logic (Default)
# Find studies by EITHER Wahl OR Vose
print("--- Searching for Wahl OR Vose ---")
search_by_investigator_or = dataset.search_studies(
    investigators=["Wahl, E.R.", "Vose, R.S."],
)
print(f"Found {len(search_by_investigator_or)} studies.")

# Example 2: "AND" Logic
# Find studies co-authored by BOTH Wahl AND Vose
# Note: We must specify the `investigators_and_or` parameter
print("\n--- Searching for Wahl AND Vose ---")
search_by_investigator_and = dataset.search_studies(
    investigators=["Wahl, E.R.", "Vose, R.S."],
    investigators_and_or="AND",
)
print(f"Found {len(search_by_investigator)} studies.")

[2025-12-11 10:28:33,981][INFO] - search_studies: Limit defaulted to 100 (PyleoTUPS).


--- Searching for Wahl OR Vose ---
Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&limit=100&investigators=Wahl%2C+E.R.%7CVose%2C+R.S.&investigatorsAndOr=or


Parsing NOAA studies: 100%|██████████| 24/24 [00:00<00:00, 6960.54it/s]
[2025-12-11 10:28:34,807][INFO] - search_studies: Limit defaulted to 100 (PyleoTUPS).


Found 24 studies.

--- Searching for Wahl AND Vose ---
Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&limit=100&investigators=Wahl%2C+E.R.%7CVose%2C+R.S.&investigatorsAndOr=and


Parsing NOAA studies: 100%|██████████| 2/2 [00:00<00:00, 1997.76it/s]

Found 2 studies.


In [4]:
dataset = pt.Dataset()

In [ ]:
search_incorrect_investigator_usage = dataset.search_studies(investigators=["E.R., Wahl"])

print(f"Found {len(search_incorrect_investigator_usage)} studies.")

[2025-12-11 10:28:35,380][INFO] - search_studies: Limit defaulted to 100 (PyleoTUPS).


Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&limit=100&investigators=E.R.%2C+Wahl


/Users/dhirenoswal/Desktop/TU corpus/PyleoTUPS/pyleotups/core/Dataset.py:452: UserWarning: No studies found for investigator(s): E.R., Wahl. NOAA expects 'LastName, Initials'. Try variations like:
  - 'LastName, Initials'
  - 'LastName'
  - 'Initials'
  warnings.warn(


""


##### Efficient Use of Geographic search parameter

You can search for studies within a specific geographic region using latitude and longitude bounds.

**Parameters:**
* `min_lat`, `max_lat`: Latitude range (Strict Valid: -90 to 90).
* `min_lon`, `max_lon`: Longitude range (Strict Valid: -180 to 180).

**Important Notes:**
1. **Integer Precision:** Decimal values (e.g., `23.5`) will be cast to integers (e.g., `23`) due to NOAA's API requiremnets.
2. **Bounding Box:** The search returns studies that fall *anywhere* within the box defined by these coordinates.

In [6]:
# Search for studies in a specific region (e.g., Tropical Pacific)
# Latitude: 10S to 10N (-10 to 10)
# Longitude: 120E to 150E (120 to 150)
dataset = pt.Dataset()

dataset.search_studies(
    min_lat=-10, max_lat=10,
    min_lon=120, max_lon=150,
    limit = 100
)

# View the location data in the summary
dataset.get_summary()

[2025-12-11 10:28:35,835][INFO] - search_studies: Limit defaulted to 100 (PyleoTUPS).


Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&limit=100&minLat=-10&maxLat=10&minLon=120&maxLon=150


Parsing NOAA studies: 100%|██████████| 100/100 [00:00<00:00, 4172.69it/s]
/Users/dhirenoswal/Desktop/TU corpus/PyleoTUPS/pyleotups/core/Dataset.py:501: UserWarning: Retrieved 100 studies, which is the specified limit. Consider increasing the limit parameter to fetch more studies.
  warnings.warn(


,StudyID,XMLID,StudyName,DataType,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding
0,10437,9101,"1,000 Year Ensemble Reconstructions of Tempera...",CLIMATE RECONSTRUCTIONS,950.0,-45.0,1000.0,1995.0,Calibration ensemble reconstructions of existi...,"[carbon cycle, sensitivity, Air Temperature Re...","David Frank, Valerie Trouet, Jan Esper, Christ...","[{'Author': 'Frank, D.C., J. Esper, C.C. Raibl...","[[{'DataTableID': '19235', 'DataTableName': 'F...",[{'fundingAgency': 'Swiss National Science Fou...
1,42679,83754,1006-year reconstruction of Rossby wavenumber-5,CLIMATE RECONSTRUCTIONS,950.0,-55.0,1000.0,2005.0,None,[Atmospheric and Oceanic Circulation Patterns ...,"Kai Kornhuber, Ellie Broadman, Valerie Trouet","[{'Author': 'Broadman, Ellie, Kai Kornhuber, I...","[[{'DataTableID': '56946', 'DataTableName': 'W...",[{'fundingAgency': 'US National Science Founda...
2,12203,10267,2000 Year Precipitation-Based Southern Oscilla...,CLIMATE RECONSTRUCTIONS,1900.0,-5.0,50.0,1955.0,Reconstruction of a precipitation-based Southe...,[Atmospheric and Oceanic Circulation Patterns ...,"Liguang Sun, Yuhong Wang, Wen Huang, Shican Qi...","[{'Author': 'Yan, H., L. Sun, Y. Wang, W. Huan...","[[{'DataTableID': '20526', 'DataTableName': 'S...",[{'fundingAgency': 'National Natural Science F...
3,22315,20388,2000 Year Tropical Rainfall Reconstructions,CLIMATE RECONSTRUCTIONS,2000.0,-50.0,-50.0,2000.0,Composite reconstruction of low latitude rainf...,[Precipitation Reconstruction],"Franziska Lechleitner, Sebastian Breitenbach, ...","[{'Author': 'Franziska A. Lechleitner, Sebasti...","[[{'DataTableID': '33444', 'DataTableName': 'L...","[{'fundingAgency': 'European Union', 'fundingG..."
4,12894,10958,9400 Year Cosmogenic Isotope Data and Solar Ac...,CLIMATE FORCING,9389.0,-27.0,-7439.0,1977.0,Records of common production rate of cosmogeni...,[Solar Forcing Reconstruction],"Irene Brunner, Marcus Christl, Hubertus Fische...","[{'Author': 'Steinhilber, F., J.A. Abreu, J. B...","[[{'DataTableID': '21230', 'DataTableName': 'T...",[{'fundingAgency': 'Swiss National Science Fou...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,5495,620,"Global Lake-Level Variations from 18,000 to 0 ...",LAKE LEVELS,20000.0,0.0,-18050.0,1950.0,Lake status determined at 1000-year intervals ...,"[hydrology, trends]","Frances Alayne Street-Perrott, None Marchand, ...","[{'Author': 'Street-Perrott, F.A., D.S. Marcha...","[[{'DataTableID': '9036', 'DataTableName': 'Qu...",[]
96,36694,77952,Global Marine Fossil 14C Data during the last ...,PALEOCEANOGRAPHY,52119.0,0.0,-50169.0,1950.0,None,None,"William Gray, Sophia Hines, Andrea Burke, Kass...","[{'Author': 'Rafter, Patrick A., William R. Gr...","[[{'DataTableID': '49382', 'DataTableName': 'G...",[{'fundingAgency': 'US National Science Founda...
97,5983,2681,Global Network for Isotopes in Precipitation(G...,OTHER COLLECTIONS,-7.0,-41.0,1957.0,1991.0,,None,"Kazimierz Rozanski, Luis Araguás-Araguás, Robe...",[],"[[{'DataTableID': '32472', 'DataTableName': 'G...",[]
98,31572,73252,"Global Ocean 25,000 Year Atmosphere-Ocean CO2 ...",PALEOCLIMATIC MODELING,25000.0,0.0,-23050.0,1950.0,Transient simulation of ocean carbonate chemis...,[carbon cycle],"Jun Shao, Lowell Stott, William Gray, Rosanna ...","[{'Author': 'Jun Shao, Lowell D. Stott, Willia...","[[{'DataTableID': '44097', 'DataTableName': 'S...",[{'fundingAgency': 'US National Science Founda...


```Note
PyleoTUPS retrieves and presents the first 100 response from NOAA. You can adjust the ```limit``` parameter to include more results

## Step 2: Browsing Search Results



Once you have performed a search, the `dataset` object holds the results. You can browse these results at three levels of granularity:
1.  **Study Level (`get_summary`)**: High-level metadata (Authors, Titles, Date).
2.  **Site/Table Level (`get_sites`)**: Detailed information about specific sites and the data tables available for them. **This is where you find the IDs needed to extract data.**
3.  **Publication Level (`get_publications`)**: Citation information for the studies.

In [7]:
# 1. View the high-level summary of the studies found

dataset = pt.Dataset()
dataset.search_studies(noaa_id=18316)

summary_df = dataset.get_summary()

display(summary_df)


[2025-12-11 10:28:37,851][INFO] - search_studies: Using identifier-only fetch (xml_id/NOAAStudyId). Other parameters will be ignored.


Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?NOAAStudyId=18316&dataPublisher=NOAA


Parsing NOAA studies: 100%|██████████| 1/1 [00:00<00:00, 1730.32it/s]


,StudyID,XMLID,StudyName,DataType,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding
0,18316,16019,"WAIS Divide Core 1,200 Year Atmospheric CO2 an...",ICE CORES,1217,11,733,1939,CO2 concentration and Stable Isotopic Composit...,[carbon cycle],"Thomas Bauska, Fortunat Joos, Alan Mix, Raphae...","[{'Author': 'Ahn, J., E. J. Brook, L. Mitchell...","[[{'DataTableID': '28694', 'DataTableName': 'W...",[{'fundingAgency': 'US National Science Founda...


### Detailed Site & Data Table Information
The `get_sites()` method flattens the hierarchy, providing a row for every data file (or "Data Table") associated with a study.

**Why is this important?**
To extract specific data later, you will need the `dataTableID`. This ID is unique to each file and allows PyleoTUPS to link metadata back to the data.

* **SiteName:** The name of the specific location (e.g., a specific ice core or lake).
* **DataTableID:** The unique identifier for the data file.


In [8]:
# 2. View detailed site and data table information
sites_df = dataset.get_sites()

# Display columns relevant for data extraction
# Note the 'dataTableID' column - we will use this in Step 3.
display(sites_df)

,DataTableID,DataTableName,TimeUnit,FileURL,Variables,FileDescription,TotalFilesAvailable,SiteID,SiteName,LocationName,Latitude,Longitude,MinElevation,MaxElevation,StudyID,StudyName
0,28694,WAIS2015d13CO2,CE,https://www.ncei.noaa.gov/pub/data/paleo/iceco...,"[Core_ID, depth_top_m, depth_bot_m, depth_m, a...",NOAA Template File,1,22909,WAIS Divide WDC05A,Continent>Antarctica,-79.47,-112.13,1759,1759,18316,"WAIS Divide Core 1,200 Year Atmospheric CO2 an..."
1,28803,WAIS2015CO2,CE,https://www.ncei.noaa.gov/pub/data/paleo/iceco...,"[Core_ID, depth_m, age_AD, co2_ppm, co2_1s_ppm...",NOAA Template File,1,22909,WAIS Divide WDC05A,Continent>Antarctica,-79.47,-112.13,1759,1759,18316,"WAIS Divide Core 1,200 Year Atmospheric CO2 an..."
2,28804,WAIS2015CO2flux,CE,https://www.ncei.noaa.gov/pub/data/paleo/iceco...,"[age_AD, Fal_15.9, Fal_84.1, Fal_median, Fao_1...",NOAA Template File,1,22909,WAIS Divide WDC05A,Continent>Antarctica,-79.47,-112.13,1759,1759,18316,"WAIS Divide Core 1,200 Year Atmospheric CO2 an..."


### Retrieving Citations
The `get_publications()` method returns:
1.  A **BibTeX** object (compatible with reference managers).
2.  A **DataFrame** of publication details (DOIs, years, journals).

**Parameters:**
* `save=True`: Saves a `.bib` file to your disk.
* `verbose=True`: Prints the BibTeX entries to the console.

In [9]:
# 3. Get citation data
# Returns a tuple: (BibTeX_Database, DataFrame)
bib_db, pubs_df = dataset.get_publications(save=False, verbose=False)

# Display the publication metadata
display(pubs_df)


,Author,Title,Journal,Year,Volume,Number,Pages,Type,DOI,URL,CitationKey,StudyID,StudyName
0,"Ahn, J., E. J. Brook, L. Mitchell, J. Rosen, J...",Atmospheric CO2 over the last 1000 years: A hi...,Global Biogeochemical Cycles,2012,26,2,NaN,publication,10.1029/2011GB004247,http://dx.doi.org/10.1029/2011GB004247,Rubino_Atmospheric_2012_18316,18316,"WAIS Divide Core 1,200 Year Atmospheric CO2 an..."
1,"Thomas K. Bauska, Fortunat Joos, Alan C. Mix, ...","Links between atmospheric carbon dioxide, the ...",Nature Geoscience,2015,8,5,383-387,publication,10.1038/NGEO2422,http://dx.doi.org/10.1038/NGEO2422,Brook_Links_2015_18316,18316,"WAIS Divide Core 1,200 Year Atmospheric CO2 an..."


In [10]:
# save the BibTeX database to a .bib file. File will be saved in the directory where the notebook/script is run.
dataset.get_publications(save=True)


/Users/dhirenoswal/Desktop/TU corpus/PyleoTUPS/pyleotups/core/Dataset.py:598: UserWarning: No path specified. Saving BibTeX to: bibtex_20251211_1028.bib
  warnings.warn(f"No path specified. Saving BibTeX to: {path}")


(BibliographyData(
   entries=OrderedCaseInsensitiveDict([
     ('Rubino_Atmospheric_2012_18316', Entry('article',
       fields=[
         ('author', 'Ahn, J., E. J. Brook, L. Mitchell, J. Rosen, J. R. McConnell, K. Taylor, D. Etheridge, M. Rubino'),
         ('title', 'Atmospheric CO2 over the last 1000 years: A high-resolution record from the West Antarctic Ice Sheet (WAIS) Divide ice core'),
         ('journal', 'Global Biogeochemical Cycles'),
         ('year', '2012'),
         ('doi', '10.1029/2011GB004247'),
         ('url', 'http://dx.doi.org/10.1029/2011GB004247')])), 
     ('Brook_Links_2015_18316', Entry('article',
       fields=[
         ('author', 'Thomas K. Bauska, Fortunat Joos, Alan C. Mix, Raphael Roth, Jinho Ahn, and Edward J. Brook'),
         ('title', 'Links between atmospheric carbon dioxide, the land carbon reservoir and climate over the past millennium'),
         ('journal', 'Nature Geoscience'),
         ('year', '2015'),
         ('doi', '10.1038/NGEO2422')

## Step 3: Extracting Data



This is the core function of PyleoTUPS. Once you have identified the data you want (from the `get_sites()` step above), you can extract it into a pandas DataFrame.

### Method A: By DataTableID (Recommended)
**Why?** When you fetch by ID, PyleoTUPS automatically links the data back to its parent Study and Site. The resulting DataFrame will contain rich metadata in its `.attrs` dictionary.

* **Input:** A string ID or a list of IDs (e.g., `["28694", "28803"]`).
* **Output:** A list of one or more pandas DataFrames.

In [11]:
# 1. View the high-level summary of the studies found

dataset = pt.Dataset()
dataset.search_studies(noaa_id=18316)

dataset.get_sites()


[2025-12-11 10:28:38,629][INFO] - search_studies: Using identifier-only fetch (xml_id/NOAAStudyId). Other parameters will be ignored.


Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?NOAAStudyId=18316&dataPublisher=NOAA


Parsing NOAA studies: 100%|██████████| 1/1 [00:00<00:00, 2832.08it/s]


,DataTableID,DataTableName,TimeUnit,FileURL,Variables,FileDescription,TotalFilesAvailable,SiteID,SiteName,LocationName,Latitude,Longitude,MinElevation,MaxElevation,StudyID,StudyName
0,28694,WAIS2015d13CO2,CE,https://www.ncei.noaa.gov/pub/data/paleo/iceco...,"[Core_ID, depth_top_m, depth_bot_m, depth_m, a...",NOAA Template File,1,22909,WAIS Divide WDC05A,Continent>Antarctica,-79.47,-112.13,1759,1759,18316,"WAIS Divide Core 1,200 Year Atmospheric CO2 an..."
1,28803,WAIS2015CO2,CE,https://www.ncei.noaa.gov/pub/data/paleo/iceco...,"[Core_ID, depth_m, age_AD, co2_ppm, co2_1s_ppm...",NOAA Template File,1,22909,WAIS Divide WDC05A,Continent>Antarctica,-79.47,-112.13,1759,1759,18316,"WAIS Divide Core 1,200 Year Atmospheric CO2 an..."
2,28804,WAIS2015CO2flux,CE,https://www.ncei.noaa.gov/pub/data/paleo/iceco...,"[age_AD, Fal_15.9, Fal_84.1, Fal_median, Fao_1...",NOAA Template File,1,22909,WAIS Divide WDC05A,Continent>Antarctica,-79.47,-112.13,1759,1759,18316,"WAIS Divide Core 1,200 Year Atmospheric CO2 an..."


In [12]:
dfs = dataset.get_data(dataTableIDs=["28694", "28803", "28804"])

In [13]:
for df in dfs:
    display(df.head())
    print(df.attrs)

,Core_ID,depth_top_m,depth_bot_m,depth_m,age_AD,d13C_CO2
0,WDC05A,84.41,84.59,84.50,1916.37,-6.868
1,WDC05A,94.82,95.00,94.91,1871.58,-6.569
2,WDC05A,100.34,100.54,100.44,1847.24,-6.542
3,WDC05A,106.06,106.24,106.15,1822.16,-6.419
4,WDC05A,109.02,109.12,109.07,1809.02,-6.420


{'variables': ['Core_ID', 'depth_top_m', 'depth_bot_m', 'depth_m', 'age_AD', 'd13C_CO2'], 'NOAAStudyId': '18316', 'StudyName': 'WAIS Divide Core 1,200 Year Atmospheric CO2 and CO2 Stable Isotope Data'}


,Core_ID,depth_m,age_AD,co2_ppm,co2_1s_ppm,num_reps
0,WDC05A,78.973,1939.7,311.6,1.6,6
1,WDC05A,80.095,1934.8,307.1,0.2,12
2,WDC05A,80.595,1933,306.5,1.3,5
3,WDC05A,82.588,1924.5,303.9,0.3,6
4,WDC05A,84.64,1915.8,298.8,1.2,4


{'variables': ['Core_ID', 'depth_m', 'age_AD', 'co2_ppm', 'co2_1s_ppm', 'num_reps'], 'NOAAStudyId': '18316', 'StudyName': 'WAIS Divide Core 1,200 Year Atmospheric CO2 and CO2 Stable Isotope Data'}


,age_AD,Fal_15.9,Fal_84.1,Fal_median,Fao_15.9,Fao_84.1,Fao_median,del_lc_stocks_15.9,del_lc_stocks_84.1,del_lc_stocks_median,del_oc_stocks_15.9,del_oc_stocks_84.1,del_oc_stocks_median
0,755,-0.0076906,0.0531275,0.0203811,-0.0687170,0.0604406,-0.0027614,-0.0076906,0.0531275,0.0203811,-0.0687170,0.0604406,-0.0027614
1,756,-0.0004563,0.0690373,0.0312458,-0.0801152,0.0490091,-0.0142406,-0.0085262,0.1224410,0.0515803,-0.1481970,0.1136900,-0.0178154
2,757,0.0047517,0.0822963,0.0399637,-0.0895057,0.0418756,-0.0238649,-0.0032509,0.2051300,0.0920047,-0.2388610,0.1500030,-0.0407658
3,758,0.0091025,0.0943422,0.0479293,-0.0989552,0.0355697,-0.0299825,0.0074808,0.2994590,0.1400970,-0.3392090,0.1874460,-0.0676734
4,759,0.0128880,0.1057800,0.0545621,-0.1078840,0.0312313,-0.0361903,0.0201656,0.4032080,0.1931260,-0.4426620,0.2180520,-0.1107660


{'variables': ['age_AD', 'Fal_15.9', 'Fal_84.1', 'Fal_median', 'Fao_15.9', 'Fao_84.1', 'Fao_median', 'del_lc_stocks_15.9', 'del_lc_stocks_84.1', 'del_lc_stocks_median', 'del_oc_stocks_15.9', 'del_oc_stocks_84.1', 'del_oc_stocks_median'], 'NOAAStudyId': '18316', 'StudyName': 'WAIS Divide Core 1,200 Year Atmospheric CO2 and CO2 Stable Isotope Data'}


NOAA has templated as well as non-templated data tables. The templated tables follow a standard format and include metadata in the table itself, while non-templated tables may have varying formats. 

In [14]:
speleo8k_ds = pt.Dataset()
speleo8k_ds.search_studies(noaa_id=9957)
speleo8k_ds.get_summary()

[2025-12-11 10:28:43,889][INFO] - search_studies: Using identifier-only fetch (xml_id/NOAAStudyId). Other parameters will be ignored.


Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?NOAAStudyId=9957&dataPublisher=NOAA


Parsing NOAA studies: 100%|██████████| 1/1 [00:00<00:00, 1269.08it/s]


,StudyID,XMLID,StudyName,DataType,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding
0,9957,8850,8.2k Event Speleothem Oxygen Isotope Data,SPELEOTHEMS,8566,7678,-6616,-5728,Oxygen isotope data from six stalagmites in Ch...,"[abrupt climate change, Intertropical Converge...","R. Lawrence Edwards, Augusto Mangini, Stephen ...","[{'Author': 'Cheng, H., D. Fleitmann, R.L. Edw...","[[{'DataTableID': '18803', 'DataTableName': 'H...",[{'fundingAgency': 'Comer Science and Educatio...


In [15]:
speleo8k_ds.get_sites()

,DataTableID,DataTableName,TimeUnit,FileURL,Variables,FileDescription,TotalFilesAvailable,SiteID,SiteName,LocationName,Latitude,Longitude,MinElevation,MaxElevation,StudyID,StudyName
0,18803,H14,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/spele...,"[age, depth, delta 18O]",Speleothem,2,31383,Hoti Cave,Continent>Asia>Western Asia>Middle East>Oman,23.08,57.35,800,800,9957,8.2k Event Speleothem Oxygen Isotope Data
1,18803,H14,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/spele...,[],Speleothem,2,31383,Hoti Cave,Continent>Asia>Western Asia>Middle East>Oman,23.08,57.35,800,800,9957,8.2k Event Speleothem Oxygen Isotope Data
2,18804,PAD07,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/spele...,"[age, depth, delta 18O]",Speleothem,2,31568,Padre Cave,Continent>South America>Brazil,-13.2167,-44.05,650,800,9957,8.2k Event Speleothem Oxygen Isotope Data
3,18804,PAD07,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/spele...,[],Speleothem,2,31568,Padre Cave,Continent>South America>Brazil,-13.2167,-44.05,650,800,9957,8.2k Event Speleothem Oxygen Isotope Data
4,18805,PX5,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/spele...,"[age, depth, delta 18O]",Speleothem,2,31569,Paixão Cave,Continent>South America>Brazil,-12.65,-41.05,650,800,9957,8.2k Event Speleothem Oxygen Isotope Data
5,18805,PX5,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/spele...,[],Speleothem,2,31569,Paixão Cave,Continent>South America>Brazil,-12.65,-41.05,650,800,9957,8.2k Event Speleothem Oxygen Isotope Data
6,18801,D4 Dongge,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/spele...,"[age, depth, delta 18O]",Speleothem,2,6554,Dongge Cave,Continent>Asia>Eastern Asia>China,25.28,108.08,680,680,9957,8.2k Event Speleothem Oxygen Isotope Data
7,18801,D4 Dongge,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/spele...,[],Speleothem,2,6554,Dongge Cave,Continent>Asia>Eastern Asia>China,25.28,108.08,680,680,9957,8.2k Event Speleothem Oxygen Isotope Data
8,18802,DA Dongge,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/spele...,"[age, depth, delta 18O]",Speleothem,2,6554,Dongge Cave,Continent>Asia>Eastern Asia>China,25.28,108.08,680,680,9957,8.2k Event Speleothem Oxygen Isotope Data
9,18802,DA Dongge,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/spele...,[],Speleothem,2,6554,Dongge Cave,Continent>Asia>Eastern Asia>China,25.28,108.08,680,680,9957,8.2k Event Speleothem Oxygen Isotope Data


In [16]:
speleo8k_ds_df = speleo8k_ds.get_data(dataTableIDs= ["18803"])

In [17]:
for df in speleo8k_ds_df:
    display(df.head())
    print(df.attrs)

,Sample Number,Depth to top (mm),238U (ppb),(ppb),232Th (ppt),(ppt),230Th/232Th (atomic x 10-6),d234U* measured,230Th/238U activity,Age (yr) uncorrected,Age (yr) corrected,d234U initial corrected,Age (yr BP)* corrected
0,D4-4,1402,356,±0.5,62,±5,6700 ±600,-26 ±1.8,0.06977 ±0.00046,8101 ±58,8091 ±58,-26.6 ±1.9,8035 ±58
1,D4-9 n,1404,388,±0.1,89,±4,5000 ±200,-23.9 ±1.2,0.07009 ±0.00014,8120 ±19,8108 ±21,-24.5 ±1.2,8052 ±21
2,D4-1,1410,497.2,±0.8,468,±6,1240 ±20,-26.3 ±1.5,0.07063 ±0.00035,8206 ±44,8157 ±54,-26.9 ±1.6,8101 ±54
3,D4-8 n,1425,493.8,±0.1,191,±5,3020 ±80,-28.8 ±0.5,0.07065 ±0.00016,8232 ±20,8212 ±24,-29.5 ±0.5,8156 ±24
4,D4-10 n,1447,370.5,±0.1,143,±5,3100 ±100,-13.2 ±0.7,0.07259 ±0.00017,8326 ±21,8306 ±25,-13.5 ±0.7,8250 ±25


{'NOAAStudyId': '9957', 'StudyName': '8.2k Event Speleothem Oxygen Isotope Data'}


,Sample Number,Depth to top (mm),238U (ppb),(ppb),232Th (ppt),(ppt),230Th/232Th (atomic x 10-6),d234U* measured,230Th/238U activity,Age (yr) uncorrected,Age (yr) corrected,d234U initial corrected,Age (yr BP)* corrected
0,DA5-13,858.5,586.3,±0.8,570,±5,1140 ±10,-69.9 ±1.7,0.06679 ±0.00033,8126 ±44,8073 ±56,-71.5 ±1.7,8018 ±56
1,DA5-6,859,545.6,±0.7,200,±10,2900 ±200,-72.9 ±1.1,0.06594 ±0.00048,8046 ±61,8025 ±63,-74.5 ±1.2,7970 ±63
2,DA5-5,865.1,615.3,±0.8,590,±10,1150 ±30,-75.3 ±1.2,0.06705 ±0.00051,8209 ±66,8156 ±75,-77.1 ±1.2,8101 ±75
3,DA-17 n,865.5,752.3,±0.1,689,±5,1210 ±10,-74.8 ±0.7,0.06703 ±0.00015,8201 ±20,8151 ±32,-76.5 ±0.7,8096 ±32
4,DA5-4,871.2,516.2,±0.8,340,±10,1710 ±60,-75.6 ±1.3,0.06747 ±0.00071,8265 ±91,8229 ±94,-77.3 ±1.4,8174 ±94


{'NOAAStudyId': '9957', 'StudyName': '8.2k Event Speleothem Oxygen Isotope Data'}


,Sample Number,Depth to top (mm),238U (ppb),(ppb),232Th (ppt),(ppt),230Th/232Th (atomic x 10-6),d234U* measured,230Th/238U activity,Age (yr) uncorrected,Age (yr) corrected,d234U initial corrected,Age (yr BP)* corrected
0,Q5-15 n,2,659.6,±0.8,250,±10,2800 ±100,-83.1 ±1.6,0.06592 ±0.00027,8138 ±37,8125 ±38,-85.1 ±1.6,8069 ±38
1,Q5-14 n,7,646,±0.7,290,±10,2500 ±100,-75.9 ±1.4,0.06692 ±0.00026,8197 ±36,8183 ±36,-77.7 ±1.4,8127 ±36
2,Q5-19 n,11.5,739.4,±0.7,870,±10,950 ±10,-71 ±1.4,0.06763 ±0.00013,8242 ±21,8205 ±28,-72.7 ±1.4,8149 ±28
3,Q5-6 n,17,492.1,±0.1,295,±4,1840 ±20,-79.3 ±0.4,0.06695 ±0.00015,8234 ±19,8215 ±21,-81.2 ±0.4,8159 ±21
4,Q5-10 n,26,593,±0.9,970,±10,681 ±9,-84.9 ±1.8,0.06758 ±0.00022,8368 ±33,8316 ±42,-86.9 ±1.8,8260 ±42


...

{'NOAAStudyId': '9957', 'StudyName': '8.2k Event Speleothem Oxygen Isotope Data'}


,Depth,Age,d18O
0,90,8087,-5.07
1,90.5,8089,-5.37
2,91,8091,-5.49
3,91.5,8093,-5.57
4,92,8095,-4.79


{'NOAAStudyId': '9957', 'StudyName': '8.2k Event Speleothem Oxygen Isotope Data'}


,Depth,Age,d18O
0,199,7782,-6.151
1,200.5,7789,-6.372
2,202,7797,-6.474
3,203.5,7804,-6.387
4,205,7811,-5.71


{'NOAAStudyId': '9957', 'StudyName': '8.2k Event Speleothem Oxygen Isotope Data'}


### Method B: By Direct File URL
If you have a direct link to a NOAA text file (e.g., from an external paper or email), you can fetch it directly without searching first.

**Limitation:** Unless the URL is already known to the `dataset` object (via a previous search), PyleoTUPS **cannot** attach study-level metadata (like Study Name or Site Location) to the DataFrame. You will get the data, but `df.attrs` may be empty or limited to the file header.

In [19]:
dataset = pt.Dataset()
df_data_by_url = dataset.get_data(file_urls="https://www.ncei.noaa.gov/pub/data/paleo/climate_forcing/trace_gases/mcelwain1995co2.txt")

/Users/dhirenoswal/Desktop/TU corpus/PyleoTUPS/pyleotups/core/Dataset.py:1067: UserWarning: Attached 'https://www.ncei.noaa.gov/pub/data/paleo/climate_forcing/trace_gases/mcelwain1995co2.txt' is not linked to any parent study; can not add metadata.
  warnings.warn(


In [21]:
for df in df_data_by_url:
    display(df.head())
    print(df.attrs) # will have empty attrs

,Species,Age,Stomatal Density (mm2) Mean,Stomatal Density (mm2) s.d.,Stomatal Density (mm2) n,s.e.,"Stomatal Dens. (by vol., mm3)",Stomatal Index Mean,Stomatal Index s.d.,n,s.e._1,RCO2
0,Aglaophyton major (Kid. and Lang),Lower Devonian,4.5,3.9,n=23,0.82,3.0,2.26,1.1,n=19,0.25,12
1,Sawdonia ornata (Daws.) Hueber,Lower Devonian,4.3,1.9,n=38,0.32,3.6,3.05,1.0,n=13,0.28,12
2,Juncus effusus L.,Recent,309.1,72.8,n=50,10.3,416.2,12.4,2.3,n=50,0.4,1
3,Psilotum nudum (L.) Beauv.,Recent,40.1,16.8,n=145,1.4,89.7,15.5,5.6,n=92,0.6,1
4,Swillingtonia denticulata Scott and Chal.,Carboniferous,787.8,158.9,n=18,37.5,n/a,19.7,4.3,n=14,1.16,1.5


{}


## Step 4: Advanced Usage

### 1. Handling Large Searches (Pagination)
By default, `search_studies` returns up to 100 results. If you need to retrieve a large number of studies, or if you want to process them in batches, you can use the `limit` and `skip` parameters.

* `limit`: The maximum number of studies to return in one request.
* `skip`: The number of studies to skip from the beginning of the result set.

In [26]:
# Initialize a clean dataset
dataset = pt.Dataset()

# Query 1: Get the first 10 results
print("--- Fetching Batch 1 (1-10) ---")
batch1 = dataset.search_studies(min_lat=60, max_lat=70, limit=10)
print(f"Current studies in dataset: {len(dataset.studies)}")
display(batch1)


[2025-12-11 10:50:30,372][INFO] - search_studies: Limit set to 10.


--- Fetching Batch 1 (1-10) ---
Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&limit=10&minLat=60&maxLat=70


Parsing NOAA studies: 100%|██████████| 10/10 [00:00<00:00, 10559.68it/s]

Current studies in dataset: 10


,StudyID,XMLID,StudyName,DataType,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding
0,10437,9101,"1,000 Year Ensemble Reconstructions of Tempera...",CLIMATE RECONSTRUCTIONS,950.0,-45.0,1000.0,1995.0,Calibration ensemble reconstructions of existi...,"[carbon cycle, sensitivity, Air Temperature Re...","David Frank, Valerie Trouet, Jan Esper, Christ...","[{'Author': 'Frank, D.C., J. Esper, C.C. Raibl...","[[{'DataTableID': '19235', 'DataTableName': 'F...",[{'fundingAgency': 'Swiss National Science Fou...
1,14193,12194,1000 Year Composite Sea Surface Temperature Re...,PALEOCEANOGRAPHY,950.0,-46.0,1000.0,1996.0,A set of reconstructions of sea surface temper...,None,"Laura Cunningham, William Austin, Karen-Luise ...","[{'Author': 'Cunningham, L.K., Austin, W.E.N.,...","[[{'DataTableID': '24146', 'DataTableName': 'P...","[{'fundingAgency': 'European Commission', 'fun..."
2,42679,83754,1006-year reconstruction of Rossby wavenumber-5,CLIMATE RECONSTRUCTIONS,950.0,-55.0,1000.0,2005.0,None,[Atmospheric and Oceanic Circulation Patterns ...,"Kai Kornhuber, Ellie Broadman, Valerie Trouet","[{'Author': 'Broadman, Ellie, Kai Kornhuber, I...","[[{'DataTableID': '56946', 'DataTableName': 'W...",[{'fundingAgency': 'US National Science Founda...
3,22031,20009,1200 Year Atlantic Multidecadal Variability an...,CLIMATE RECONSTRUCTIONS,1150.0,-60.0,800.0,2010.0,Summer (May-September) Atlantic Multidecadal V...,[Atmospheric and Oceanic Circulation Patterns ...,"Jianglin Wang, Bao Yang, Fredrik Ljungqvist, J...","[{'Author': 'Jianglin Wang, Bao Yang, Fredrik ...","[[{'DataTableID': '33108', 'DataTableName': 'W...",[{'fundingAgency': 'National Natural Science F...
4,12203,10267,2000 Year Precipitation-Based Southern Oscilla...,CLIMATE RECONSTRUCTIONS,1900.0,-5.0,50.0,1955.0,Reconstruction of a precipitation-based Southe...,[Atmospheric and Oceanic Circulation Patterns ...,"Liguang Sun, Yuhong Wang, Wen Huang, Shican Qi...","[{'Author': 'Yan, H., L. Sun, Y. Wang, W. Huan...","[[{'DataTableID': '20526', 'DataTableName': 'S...",[{'fundingAgency': 'National Natural Science F...
5,31932,73613,231Pa/230Th Ratios in Arctic Ocean Surface Sed...,PALEOCEANOGRAPHY,NaN,NaN,NaN,NaN,"Provided Keywords: protactinium-231, 231Pa, th...",None,"Lauren Kipp, Jerry McManus, Markus Kienast",[],"[[{'DataTableID': '44537', 'DataTableName': 'A...",[]
6,14190,12191,600 Year Ensemble Climate Reconstructions and ...,CLIMATE RECONSTRUCTIONS,550.0,-61.0,1400.0,2011.0,"Ensemble Climate Reconstructions, input data f...",[Air Temperature Reconstruction],"Martin Tingley, Peter Huybers","[{'Author': 'Tingley, M.P. and P. Huybers', 'T...","[[{'DataTableID': '24136', 'DataTableName': 'T...",[{'fundingAgency': 'US National Science Founda...
7,12894,10958,9400 Year Cosmogenic Isotope Data and Solar Ac...,CLIMATE FORCING,9389.0,-27.0,-7439.0,1977.0,Records of common production rate of cosmogeni...,[Solar Forcing Reconstruction],"Irene Brunner, Marcus Christl, Hubertus Fische...","[{'Author': 'Steinhilber, F., J.A. Abreu, J. B...","[[{'DataTableID': '21230', 'DataTableName': 'T...",[{'fundingAgency': 'Swiss National Science Fou...
8,14730,12691,A 3000-year varved record of glacier activity ...,PALEOLIMNOLOGY,2976.0,-52.0,-1026.0,2002.0,None,"[Medieval Warm Period, Little Ice Age (LIA), A...","Darren Larsen, Gifford Miller, Áslaug Geirsdót...","[{'Author': 'Larsen, D.J., Miller, G.H., Geirs...","[[{'DataTableID': '24775', 'DataTableName': 'H...",[]
9,15675,13460,"A 37,000-year record of paleomagnetic and envi...",PALEOLIMNOLOGY,NaN,NaN,NaN,NaN,None,"[Arctic, temperature, precipitation]","Joseph Stoner, Mark Abbott, Jason Dorfman","[{'Author': 'Dorfman, J.M.', 'Title': 'A 37,00...","[[{'DataTableID': '25653', 'DataTableName': 'p...",[{'fundingAgency': 'US National Science Founda...


In [28]:
# Query 2: Get the next 10 results (skip the first 10)
# Note: This *overwrites* the previous search results in the 'dataset' object as we are reusing the same dataset instance. 
print("\n--- Fetching Batch 2 (11-20) ---")
batch2 = dataset.search_studies(min_lat=60, max_lat=70, limit=10, skip=10)
print(f"Current studies in dataset: {len(dataset.studies)}")
display(batch2)


[2025-12-11 10:50:47,008][INFO] - search_studies: Limit set to 10.



--- Fetching Batch 2 (11-20) ---
Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&limit=10&skip=10&minLat=60&maxLat=70


Parsing NOAA studies: 100%|██████████| 10/10 [00:00<00:00, 5585.70it/s]

Current studies in dataset: 10


,StudyID,XMLID,StudyName,DataType,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding
0,12402,10446,A Bayesian ANOVA Scheme for Calculating Climat...,OTHER COLLECTIONS,-11,-40,1961,1990,Matlab code for two-factor (location and year)...,None,Martin Tingley,"[{'Author': 'Tingley, M.P.', 'Title': 'A Bayes...","[[{'DataTableID': '20755', 'DataTableName': 'A...",[]
1,8610,2893,"ACT1,ACT3,ACT4,ACT2 - Melt Layer Thickness",ICE CORES,178,-53,1772,2003,None,None,Sarah Das,[],"[[{'DataTableID': '12448', 'DataTableName': 'A...",[]
2,6177,2209,ACT2 - Toxic Heavy Metals Data,ICE CORES,178,-52,1772,2002,None,None,"Joseph McConnell, Ross Edwards","[{'Author': 'McConnell, J.R. and R. Edwards', ...","[[{'DataTableID': '9477', 'DataTableName': 'AC...",[]
3,16279,14005,Abrupt Holocene climate transitions in the nor...,PALEOLIMNOLOGY,10300,-56,-8350,2006,"Keywords - Iceland, Lake sediment, Holocene pa...","[Arctic, abrupt climate change, Little Ice Age...","Áslaug Geirsdóttir, Gifford Miller, Darren Lar...","[{'Author': 'Geirsdóttir Á., G.H. Miller, D.J....","[[{'DataTableID': '26370', 'DataTableName': 'H...",[{'fundingAgency': 'US National Science Founda...
4,23930,22076,Ackerman - Brooks Range Upland - SAPC - ITRDB ...,TREE RING,-16,-65,1966,2015,NOAA Template Raw Measurements file added 2019...,"[thin red willow, diamondleaf willow, Salix pu...","Daniel Ackerman, R. Daniel Griffin, Sarah Hobb...","[{'Author': 'Daniel E. Ackerman, Daniel Griffi...","[[{'DataTableID': '35733', 'DataTableName': 'A...",[]
5,23931,22077,Ackerman - Inigok Riparian - SAPC - ITRDB AK162,TREE RING,-18,-66,1968,2016,NOAA Template Raw Measurements file added 2019...,"[SAPC, Salix pulchra Cham., Tealeaf Willow, di...","Daniel Ackerman, R. Daniel Griffin, Sarah Hobb...","[{'Author': 'Daniel E. Ackerman, Daniel Griffi...","[[{'DataTableID': '35734', 'DataTableName': 'A...",[]
6,23932,22075,Ackerman - Inigok Upland - SAPC - ITRDB AK163,TREE RING,-24,-66,1974,2016,NOAA Template Raw Measurements file added 2019...,"[Tealeaf Willow, thin red willow, Salix pulchr...","Daniel Ackerman, R. Daniel Griffin, Sarah Hobb...","[{'Author': 'Daniel E. Ackerman, Daniel Griffi...","[[{'DataTableID': '35735', 'DataTableName': 'A...",[]
7,23933,22074,Ackerman - Itkillik Upland - SAPC - ITRDB AK164,TREE RING,-12,-66,1962,2016,NOAA Template Raw Measurements file added 2019...,"[diamondleaf willow, thin red willow, Tealeaf ...","Daniel Ackerman, R. Daniel Griffin, Sarah Hobb...","[{'Author': 'Daniel E. Ackerman, Daniel Griffi...","[[{'DataTableID': '35736', 'DataTableName': 'A...",[]
8,22053,20037,Ackerman - Kuparuk Riparian - SAPC - ITRDB AK151,TREE RING,-22,-65,1972,2015,Each sample represents mean of 4 radii measure...,"[thin red willow, SAPC, Tealeaf Willow, diamon...","Daniel Ackerman, R. Daniel Griffin, Sarah Hobb...","[{'Author': 'Daniel Ackerman, Daniel Griffin, ...","[[{'DataTableID': '33151', 'DataTableName': 'A...",[{'fundingAgency': 'US National Science Founda...
9,22054,20039,Ackerman - Kuparuk Upland - SAPC - ITRDB AK152,TREE RING,-15,-65,1965,2015,Each sample represents mean of 4 radii measure...,"[SAPC, Salix pulchra Cham., Tealeaf Willow, th...","Daniel Ackerman, R. Daniel Griffin, Sarah Hobb...","[{'Author': 'Daniel Ackerman, Daniel Griffin, ...","[[{'DataTableID': '33152', 'DataTableName': 'A...",[{'fundingAgency': 'US National Science Founda...


### 2. Combining Datasets


A powerful feature of PyleoTUPS is the ability to merge `Dataset` objects. This allows you to construct complex collections of data that might be difficult to express in a single search query.

For example, you can perform two separate searches (e.g., one for "Pollen" and one for "Ice Cores") and combine them into a single dataset for analysis.

* `+` operator: Creates a new Dataset containing the union of two others.
* `+=` operator: Adds results from one Dataset into another (in-place).

In [34]:
# Create two separate datasets
ds_pollen = pt.Dataset()
ds_ice = pt.Dataset()

# Search 1: Pollen studies in a specific region
print("Searching for Pollen...")
search_pollen = ds_pollen.search_studies(keywords=["pollen"], min_lat=40, max_lat=50, limit=5)
display(search_pollen[['StudyName', 'StudyID']])

# Search 2: Ice Core studies in the same region
print("Searching for Ice Cores...")
search_ice = ds_ice.search_studies(keywords=["ice core"], min_lat=40, max_lat=50, limit=5)
display(search_ice[['StudyName', 'StudyID']])

# Combine them into a master dataset
ds_combined = ds_pollen + ds_ice

print(f"\nPollen studies: {len(ds_pollen.studies)}")
print(f"Ice Core studies: {len(ds_ice.studies)}")
print(f"Combined total: {len(ds_combined.studies)}")

# Verify the combination by looking at the summary
ds_combined.get_summary()[['StudyName', 'StudyID']].head(10)

[2025-12-11 10:53:15,987][INFO] - search_studies: Limit set to 5.


Searching for Pollen...
Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&limit=5&keywords=pollen&minLat=40&maxLat=50


Parsing NOAA studies: 100%|██████████| 5/5 [00:00<00:00, 15252.01it/s]
/Users/dhirenoswal/Desktop/TU corpus/PyleoTUPS/pyleotups/core/Dataset.py:501: UserWarning: Retrieved 5 studies, which is the specified limit. Consider increasing the limit parameter to fetch more studies.
  warnings.warn(


,StudyName,StudyID
0,Central Nova Scotia Pollen Temperature and Pre...,37245
1,"Doner 2009 Bear Lake, Utah-Idaho 19ka Pollen Data",8711
2,Orbital- and millennial-scale vegetation and c...,6187
3,Williams et al. 2004 Late Quaternary North Ame...,5973
4,Williams et al. 2006 North American Pollen Atlas,5974


[2025-12-11 10:53:16,630][INFO] - search_studies: Limit set to 5.


Searching for Ice Cores...
Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&limit=5&keywords=ice+core&minLat=40&maxLat=50


Parsing NOAA studies: 100%|██████████| 5/5 [00:00<00:00, 2753.61it/s]
/Users/dhirenoswal/Desktop/TU corpus/PyleoTUPS/pyleotups/core/Dataset.py:501: UserWarning: Retrieved 5 studies, which is the specified limit. Consider increasing the limit parameter to fetch more studies.
  warnings.warn(


,StudyName,StudyID
0,"Altai Ice Cores, Siberian and Mongolian Altai,...",40942
1,"Altai, Siberia 750 Year Ice Core d18O and Temp...",12885
2,Alto dell'Ortles Glacier Ice Core Age Models,20367
3,"Antarctica Station Dome Concordia Chemistry, S...",40399
4,"Atmospheric CH4, Interpolar Difference and Exc...",38619



Pollen studies: 5
Ice Core studies: 5
Combined total: 10


,StudyName,StudyID
0,Central Nova Scotia Pollen Temperature and Pre...,37245
1,"Doner 2009 Bear Lake, Utah-Idaho 19ka Pollen Data",8711
2,Orbital- and millennial-scale vegetation and c...,6187
3,Williams et al. 2004 Late Quaternary North Ame...,5973
4,Williams et al. 2006 North American Pollen Atlas,5974
5,"Altai Ice Cores, Siberian and Mongolian Altai,...",40942
6,"Altai, Siberia 750 Year Ice Core d18O and Temp...",12885
7,Alto dell'Ortles Glacier Ice Core Age Models,20367
8,"Antarctica Station Dome Concordia Chemistry, S...",40399
9,"Atmospheric CH4, Interpolar Difference and Exc...",38619


## STAY TUNED FOR MORE:

Check the upcoming releases for:
- Data Extraction from Excel Files
- Dataset access from PANAGAEA